# 04 — End-to-End Conformance Scenario F6: Coexistence, Agreements & Revocation

This notebook executes the **complete 29-step normative Scenario F6 (`F6.1`–`F6.4`)** across two and a half months of simulated time (`2026-09-01` → `2026-11-12`) in a few seconds using the shared `clock_file`:

- **`F6.1` Namespace & Vocabulary Setup (`2026-09-01T09:00:00Z`)**: Create `demo`, assign steward `sergio`, register proxy vocabulary `dcat-ap-context` (`r/1`).
- **`F6.2` Initial & Breaking Releases**: Publish `1.0.0` (`odrl:use`, `ex:access`), verify `grace_window: "P3D"` on `2.0.0` is rejected with `422 DATES_GRACE_TOO_SHORT`, and approve `2.0.0` (`odrl:use`, `odrl:read`, coexistence `[2026-10-03T10:00:00Z, 2026-11-02T10:00:00Z)`).
- **`F6.3` Coexistence, Lease Margin (`L = PT10S`) & Sunset Header**: Evaluate `m1` (`odrl:use`), `m2` (`ex:access`), and `m3` (`odrl:read`) before `new_content_from`, inside the `-5s` lease margin, during coexistence (`VHub-Verdict: warning;...;sunset=@1793613600`), inside the `+5s` post-enforcement lease margin, and after retirement.
- **`F6.4` Agreement Continuity (`urn:uuid:A1`), Package Retention & Revocation**: Capture `A1` under `1.0.0`, verify `1.0.0` stays `retained` for `A1` transfers after `1.0.0` retires for new intake, revoke `1.0.0`, observe `AGREEMENT_REVOCATION_SCHEDULED`, and verify transfer rejection (`400 AGREEMENT_VERSION_REVOKED`) once `revoked_from + L` elapses!

In [ ]:
import time
from lab import AlexandriaLab, show_events_table, show_f6_matrix, show_state_machine

lab = AlexandriaLab(namespace="demo", compiler="poc", initial_clock="2026-09-01T09:00:00Z")
lab.start_hub_stack()

ana = lab.token("ana", admin=True)
sergio = lab.token("sergio")
rows: list[dict] = []


def rec(step: str, status: int, code: str = "", verdict: str = "") -> None:
    rows.append({"at": lab._current_clock, "step": step, "status": status, "code": code, "verdict": verdict})


def f6_draft(version: str) -> dict:
    enum = ["odrl:use", "ex:access"] if version == "1.0.0" else ["odrl:use", "odrl:read"]
    return {
        "version": version,
        "profiles": [f"{lab.hub_url}/pub/demo/profiles/odrl/{version}"],
        "slots": [{"slot": "policy", "obligation": "mandatory", "applies_to": ["policy_intake"]}],
        "files": [
            {
                "path": "schemas/policy.schema.json",
                "media_type": "application/schema+json",
                "role": "json-schema",
                "slot": "policy",
                "applies_to": ["policy_intake"],
                "content": {"type": "object", "properties": {"action": {"enum": enum}}, "required": ["action"]},
            }
        ],
    }

## Phase `F6.1` & `F6.2` — Governance Setup, `1.0.0` Initial & `2.0.0` Breaking Release

In [ ]:
# F6.1: Namespace, Steward, Proxy Vocabulary
st, _, _ = lab.request("POST", f"{lab.hub_url}/v1/namespaces", token=ana, body={"slug": "demo"}, verbose=False)
rec("F6.1 create namespace demo", st)

st, _, _ = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/stewards", token=ana, body={"iss": lab.idp_url, "sub": "sergio"}, verbose=False)
rec("F6.1 assign steward sergio", st)

st, _, voc = lab.request(
    "POST", f"{lab.hub_url}/v1/namespaces/demo/vocabularies",
    token=sergio,
    body={"name": "dcat-ap-context", "kind": "proxy", "media_type": "application/ld+json", "upstream": f"{lab.conn_url}/dcat-ap.jsonld"},
    verbose=False,
)
rev_label = "r/" + voc["revisions"][0]["id"].rsplit("/", 1)[-1]
rec("F6.1 proxy vocabulary revision", st, code=rev_label)

# F6.2: Approve 1.0.0 at 2026-09-01T10:00:00Z
lab.set_clock("2026-09-01T10:00:00Z")
_, hdr_d1, _ = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts", token=sergio, body=f6_draft("1.0.0"), verbose=False)
d1_id = hdr_d1["location"].rsplit("/", 1)[-1]
_, _, c1 = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts/{d1_id}:propose", token=sergio, verbose=False)
c1_id = c1["id"].rsplit("/", 1)[-1]
st, _, r1 = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/candidates/{c1_id}:approve", token=sergio)
rec("F6.2 approve 1.0.0", st, code=r1["credentialSubject"]["class"])

# Start alexandria-client & switch contract-agreement-in + transfer-request to enforce
lab.start_client()
lab.wait_synced("2026-09-01T10:00:00Z")
for rid in ("contract-agreement-in", "transfer-request"):
    _, hr, rule_obj = lab.request("GET", f"{lab.admin_url}/rules/{rid}", verbose=False)
    rule_obj["mode"] = "enforce"
    lab.request("PUT", f"{lab.admin_url}/rules/{rid}", headers={"If-Match": hr["etag"]}, body=rule_obj, verbose=False)

# F6.2: Propose 2.0.0 at 2026-10-01T10:00:00Z, reject P3D grace window, then approve default P30D grace
lab.set_clock("2026-10-01T10:00:00Z")
_, hdr_d2, _ = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts", token=sergio, body=f6_draft("2.0.0"), verbose=False)
d2_id = hdr_d2["location"].rsplit("/", 1)[-1]
_, _, c2 = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts/{d2_id}:propose", token=sergio, verbose=False)
c2_id = c2["id"].rsplit("/", 1)[-1]

st, _, err_p3d = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/candidates/{c2_id}:approve", token=sergio, body={"grace_window": "P3D"})
rec("F6.2 approve 2.0.0 grace P3D", st, code=err_p3d["code"])

st, _, r2 = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/candidates/{c2_id}:approve", token=sergio)
cs2 = r2["credentialSubject"]
rec("F6.2 approve 2.0.0", st, code=f"{cs2['class']} {cs2['new_content_from']} {cs2['enforced_from']}")
lab.wait_synced("2026-10-01T10:00:00Z")

## Phase `F6.3` & `F6.4` — Coexistence Matrix, Agreement `A1` Retention & Revocation

In [ ]:
try:
    def eval_m1_m3(verbose_m2: bool = False) -> None:
        for step, action in [("F6.3 m1 odrl:use", "odrl:use"), ("F6.3 m2 ex:access", "ex:access"), ("F6.3 m3 odrl:read", "odrl:read")]:
            st, hdr, body = lab.request(
                "POST", f"{lab.dsp_ingress_url}/negotiations/urn:uuid:p-1/request",
                body={"@type": "ContractRequestMessage", "providerPid": "urn:uuid:p-1", "consumerPid": "urn:uuid:c-1", "action": action},
                verbose=(verbose_m2 and action == "ex:access"),
            )
            rec(step, st, code=(body.get("code", "") if st >= 400 and isinstance(body, dict) else ""), verdict=hdr.get("vhub-verdict", ""))

    # 1. 2026-10-02T12:00:00Z: 2.0.0 is Scheduled -> only 1.0.0 in force
    lab.set_clock("2026-10-02T12:00:00Z", silent=True)
    eval_m1_m3()

    # 2. 2026-10-03T09:59:55Z: 5s before new_content_from(2.0.0), inside lease_margin L=10s -> coexistence begins!
    lab.set_clock("2026-10-03T09:59:55Z", silent=True)
    eval_m1_m3()

    # 3. 2026-10-10T12:00:00Z: Mid-coexistence (Transition state), :explain m2, and capture Agreement A1
    lab.set_clock("2026-10-10T12:00:00Z")
    _, _, st_trans = lab.request("GET", f"{lab.admin_url}/status", verbose=False)
    show_state_machine(st_trans["state"], subtitle="1.0.0 + 2.0.0 coexisting in V_t")
    eval_m1_m3(verbose_m2=True)

    st, _, exp_m2 = lab.request(
        "POST", f"{lab.admin_url}/rules:explain",
        body={
            "listener": "dsp-ingress",
            "method": "POST",
            "path": "/negotiations/urn:uuid:p-1/request",
            "body": {"@type": "ContractRequestMessage", "providerPid": "urn:uuid:p-1", "consumerPid": "urn:uuid:c-1", "action": "ex:access"},
        },
        verbose=False,
    )
    rec("F6.3 :explain m2", st, code=exp_m2["action"], verdict=exp_m2.get("vhub_verdict", ""))

    st, hdr_a1, _ = lab.request(
        "POST", f"{lab.dsp_ingress_url}/negotiations/urn:uuid:c-1/agreement",
        body={"@type": "ContractAgreementMessage", "providerPid": "urn:uuid:p-1", "consumerPid": "urn:uuid:c-1", "action": "ex:access", "agreement": {"@id": "urn:uuid:A1"}},
    )
    _, _, a1_doc = lab.request("GET", f"{lab.admin_url}/agreements/urn:uuid:A1")
    rec("F6.4 A1 agreement ex:access", st, code=a1_doc["package_version"], verdict=hdr_a1.get("vhub-verdict", ""))

    # 4. 2026-11-02T10:00:05Z: 5s after enforced_from(2.0.0), still inside lease_margin L=10s
    lab.set_clock("2026-11-02T10:00:05Z", silent=True)
    eval_m1_m3()

    # 5. 2026-11-05T12:00:00Z: 1.0.0 retired for new intake (m2 rejected!), but retained locally for Agreement A1 transfers
    lab.set_clock("2026-11-05T12:00:00Z")
    eval_m1_m3()

    st, _, v_list = lab.request("GET", f"{lab.admin_url}/versions", verbose=False)
    role_100 = next(v["local_role"] for v in v_list if v["version"] == "1.0.0")
    rec("F6.4 1.0.0 package retained", st, code=role_100)

    msg_transfer = {"@type": "TransferRequestMessage", "consumerPid": "urn:uuid:t-1", "agreementId": "urn:uuid:A1"}
    st, hdr_tr1, _ = lab.request("POST", f"{lab.dsp_ingress_url}/transfers/request", body=msg_transfer, verbose=False)
    rec("F6.4 A1 transfer", st, verdict=hdr_tr1.get("vhub-verdict", ""))

    # 6. 2026-11-10T00:00:00Z: Steward revokes 1.0.0 -> revoked_from = 2026-11-12T00:00:00Z
    lab.set_clock("2026-11-10T00:00:00Z")
    st, _, rev_vc = lab.request(
        "POST", f"{lab.hub_url}/v1/namespaces/demo/versions/1.0.0:revoke",
        token=sergio, body={"justification": "Critical policy vulnerability in ex:access"},
    )
    rec("F6.4 revoke 1.0.0", st, code=rev_vc["credentialSubject"]["revoked_from"])
    lab.wait_synced("2026-11-10T00:00:00Z")

    # Wait briefly for AGREEMENT_REVOCATION_SCHEDULED event
    deadline = time.time() + 3.0
    ev_found = False
    while time.time() < deadline:
        if any(e.get("code") == "AGREEMENT_REVOCATION_SCHEDULED" for e in lab.read_events()):
            ev_found = True
            break
        time.sleep(0.05)
    assert ev_found
    rec("F6.4 client event", 200, code="AGREEMENT_REVOCATION_SCHEDULED")

    # 7. 2026-11-12T00:00:05Z (within L=10s -> 200) vs 2026-11-12T00:00:10Z (revoked_from + L -> 400 AGREEMENT_VERSION_REVOKED)
    lab.set_clock("2026-11-12T00:00:05Z", silent=True)
    st, _, _ = lab.request("POST", f"{lab.dsp_ingress_url}/transfers/request", body=msg_transfer, verbose=False)
    rec("F6.4 A1 transfer", st)

    lab.set_clock("2026-11-12T00:00:10Z")
    st, hdr_tr_rev, body_tr_rev = lab.request("POST", f"{lab.dsp_ingress_url}/transfers/request", body=msg_transfer)
    rec("F6.4 A1 transfer", st, code=body_tr_rev["code"], verdict=hdr_tr_rev.get("vhub-verdict", ""))

    assert len(rows) == 29, f"Expected 29 F6 rows, got {len(rows)}"
    show_f6_matrix(rows, title="Complete 29-Row Normative Scenario F6 (F6.1–F6.4)")
    show_events_table(lab.read_events(), title="Scenario F6 Client Event Stream")
finally:
    lab.close()